# Coleta de Dados de informações dos jogos

In [1]:
# Importando Bibliotecas de conexão
import mysql.connector
import selenium

In [ ]:
# Acessando o banco de dados
db = mysql.connector.connect(
    host = "localhost",
    user = "root",
    password = "raphael",
    database = "World_Cup_26"
)

cursor = db.cursor()

cursor.execute("SELECT * FROM seleções")

for x in cursor:
  print(x)

In [ ]:
cursor.execute("SELECT * FROM seleções")
dados = cursor.fetchall()

In [4]:
# Lidar com Tempo de espera
import time
from time import sleep

############################### SELENIUM ###############################
from selenium import webdriver # navegador

# Ações ###############################
from selenium.webdriver.common.by import By # localizar elementos
from selenium.webdriver.common.keys import Keys # comandos do teclado
from selenium.webdriver.common.action_chains import ActionChains # ações do mouse e teclado

# Exceções|Erros e Espera ###############################
from selenium.common.exceptions import NoSuchElementException # exceção para elementos não encontrados # CONTROLE DE ERROS
from selenium.webdriver.support.ui import WebDriverWait # esperar

# condições de espera ###############################
from selenium.webdriver.support.expected_conditions import (visibility_of, staleness_of, invisibility_of_element, visibility_of_element_located)
from selenium.webdriver.support import expected_conditions as EC # condições de espera (atalho)
from selenium.common.exceptions import TimeoutException # exceção para tempo limite

## ChromeDriver ###############################
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
browser = webdriver.Chrome(service=Service(ChromeDriverManager().install()))

browser.get('https://www.sofascore.com/pt/football/team/brazil/4748#tab:statistics')


In [5]:
# Requisições HTTP
import requests
from bs4 import BeautifulSoup
import pyautogui
import pyperclip
import re

In [ ]:
# # FUNÇÃO: ENCONTRAR LOCALIZAÇÃO 
# def achar_loc():
#     pyautogui.sleep(2)
#     posicao = pyautogui.position()
#     texto = 'click_and_reset(' + str(posicao[0]) + ', ' + str(posicao[1]) + ')'
#     pyperclip.copy(texto)
#     return texto

# # FUNÇÃO: CLICAR E RETONAR
# def click_and_reset(x, y):
#     posicao_antiga = pyautogui.position()
#     pyautogui.click(x, y)
#     pyautogui.moveTo(posicao_antiga)

# # FUNÇÃO: VERIFICAR CONTORNO COM O MOUSE
# def verificar_quadratura(left, top, width, height):
#     tempo_sleep = 0.1
#     for _ in range(6):
#         pyautogui.moveTo(left, top)
#         sleep(tempo_sleep)
#         pyautogui.moveTo(left, height)
#         sleep(tempo_sleep)
#         pyautogui.moveTo(width, height)
#         sleep(tempo_sleep)
#         pyautogui.moveTo(width, top)

In [ ]:
# pasta_gpt = 'C:/Users/raphael.eugenio/Desktop/Raphael/imagens/'

# left = 900
# top = 190

# width = 300 - left
# height = 915 - top

# regiao = (left, top, width, height)
# verificar_quadratura(left, top, width+left, height+top)

# res = pyautogui.locateOnScreen(pasta_gpt+'conmebol.png', region = regiao, confidence = 0.80)
print(res)


ValueError: needle dimension(s) exceed the haystack image or region dimensions

# Pegando um caso

In [ ]:
pais = 'Noruega'

In [ ]:
print(pais)

# Barra pesquisar
pesquisa = browser.find_element("xpath", '//*[@id="search-trigger-button"]/div')
pesquisa.click()

# Digitar a seleção
try:
    digite = browser.find_element("xpath", '//*[@id="search-input"]')
    digite.click()
    digite.send_keys(pais)
    print('Selenium')
except:
    pyautogui.click(410, 137)
    pyautogui.write(pais)
    time.sleep(2)
    # print('pyautogui')

# Selecionar o pais
try:
    selecione = browser.find_element('xpath', '/html/body/div[6]/div[2]/div/div[2]/div/div[1]/a/div')
    selecione.click()
    selecione.click()
    print('Selenium')
except:
    pyautogui.click(377, 271)
    time.sleep(2)
    # print('pyautogui')

# Digitar botão de estatisticas
try:
    estatisticas = browser.find_element("xpath", '//*[@id="horizontally-scrollable«re8»"]/div/div/div/button[2]')
    estatisticas.click()
except:
    pyautogui.click(x=742, y=423)
    time.sleep(1)

# Copiando toda a pagina
pyautogui.hotkey('ctrl', 'a')
time.sleep(0.5)
pyautogui.hotkey('ctrl', 'c')
texto = pyperclip.paste()

# Filltrando
parte = texto.split("Geral")[1].split("Cartões vermelhos")[0]

# Limpando o texto
linhas = [l.strip() for l in parte.split('\n') if l.strip()]

# Transformando em Dicionario
dados = {}

for i in range(0, len(linhas)-1, 2):
    chave = linhas[i]
    valor = linhas[i+1]
    dados[chave] = valor

print(dados)

In [ ]:
pyautogui.position()

# Executando

In [ ]:
def tratar_valor(valor):
    """Limpa strings como '61.3%', '163 (75%)' ou '2/4' para salvar apenas números."""
    if not valor: 
        return 0
    # Extrai apenas o primeiro número (inteiro ou decimal) da string
    resultado = re.findall(r"[-+]?\d*\.\d+|\d+", str(valor))
    return float(resultado[0]) if resultado else 0

def salvar_estatisticas(id_selecao, stats):
    # Mapeamento: Chave do site -> Coluna da sua tabela MySQL
    mapas = {
        "estatisticas_geral": {
            "Partidas": "Partidas", 
            "Gols marcados": "Gols_marcados", 
            "Gols sofridos": "Gols_sofridos", 
            "Assistências": "Assistências"
        },
        "estatisticas_ataque": {
            "Gols por partida": "Gols_por_partida", 
            "Gols de pênalti": "Gols_de_pênalti",
            "Gols de falta": "Gols_de_falta", 
            "Gols de dentro da área": "Gols_de_dentro_da_área",
            "Gols de fora da área": "Gols_de_fora_da_área", 
            "Gols com a perna esquerda": "Gols_com_a_perna_esquerda",
            "Gols com a perna direita": "Gols_com_a_perna_direita", 
            "Gols de cabeça": "Gols_de_cabeça",
            "Grandes chances de gol por jogo": "Grandes_chances_de_gol_por_jogo",
            "Grandes chances perdidas por jogo": "Grandes_chances_perdidas_por_jogo",
            "Total de finalizações por jogo": "Total_de_finalizações_por_jogo",
            "Chutes certos por jogo": "Chutes_certos_por_jogo", 
            "Chutes errados por jogo": "Chutes_errados_por_jogo",
            "Chutes bloqueados por jogo": "Chutes_bloqueados_por_jogo", 
            "Dribles certos por jogo": "Dribles_certos_por_jogo",
            "Escanteios por jogo": "Escanteios_por_jogo", 
            "Faltas (Tiros Diretos) por jogo": "Faltas_Tiros_Diretos_por_jogo",
            "Finalizações na trave": "Finalizações_na_trave"
        },
        "estatisticas_passes": {
            "Contra-ataques": "Contra_ataques", 
            "Posse de bola": "Posse_de_bola",
            "Passes certos": "Passes_certos", 
            "Passes no próprio campo": "Passes_no_próprio_campo",
            "Passes certos no terço final": "Passes_certos_no_terço_final",
            "Bolas longas": "Bolas_longas", 
            "Cruzamentos certos": "Cruzamentos_certos"
        },
        "estatisticas_defesa": {
            "Jogos sem sofrer gols": "Jogos_sem_sofrer_gols", 
            "Gols sofridos por jogo": "Gols_sofridos_por_jogo",
            "Desarmes por jogo": "Desarmes_por_jogo", 
            "Interceptações por jogo": "Interceptações_por_jogo",
            "Cortes por jogo": "Cortes_por_jogo", 
            "Defesas por jogo": "Defesas_por_jogo",
            "Bolas recuperadas por jogo": "Bolas_recuperadas_por_jogo",
            "Erros que levaram à finalização": "Erros_que_levaram_à_finalização",
            "Erros que levaram ao gol": "Erros_que_levaram_ao_gol", 
            "Pênaltis cometidos": "Pênaltis_cometidos",
            "Gols de pênalti concedidos": "Gols_de_pênalti_concedidos", 
            "Tirar em cima da linha": "Tirar_em_cima_da_linha"
        },
        "estatisticas_outros": {
            "Último homem a desarmar": "Último_homem_a_desarmar", 
            "Desarmes por partida": "Desarmes_por_partida",
            "Duelos ganhos pelo chão": "Duelos_ganhos_pelo_chão", 
            "Duelos aéreos ganhos": "Duelos_aéreos_ganhos",
            "Perda da posse de bola por jogo": "Perda_da_posse_de_bola_por_jogo", 
            "Laterais por jogo": "Laterais_por_jogo",
            "Tiros de meta por jogo": "Tiros_de_meta_por_jogo", 
            "Impedimentos por jogo": "Impedimentos_por_jogo",
            "Faltas por jogo": "Faltas_por_jogo", 
            "Cartões amarelos por partida": "Cartões_amarelos_por_partida",
            "Cartões vermelhos": "Cartões_vermelhos"
        }
    }

    # Percorre cada tabela do banco de dados definida no mapa
    for tabela, colunas_mapa in mapas.items():
        dados_inserir = {"id_selecao": id_selecao}
        
        # Filtra o dicionário 'stats' capturado para pegar só o que interessa para esta tabela
        for chave_site, coluna_banco in colunas_mapa.items():
            if chave_site in stats:
                dados_inserir[coluna_banco] = tratar_valor(stats[chave_site])
        
        # Se encontrou estatísticas para essa tabela, faz o INSERT
        if len(dados_inserir) > 1:
            cols = ", ".join(dados_inserir.keys())
            vals = ", ".join(["%s"] * len(dados_inserir))
            sql = f"INSERT INTO {tabela} ({cols}) VALUES ({vals})"
            
            try:
                cursor.execute(sql, list(dados_inserir.values()))
            except Exception as e:
                print(f"Erro ao inserir na tabela {tabela}: {e}")
    
    # Comita as alterações após rodar todas as tabelas para aquela seleção
    db.commit()

In [ ]:
for pais in dados:
    print(f"Iniciando: {pais[1]}")
        
    # --- PARTE 1: NAVEGAÇÃO ---
    pesquisa = browser.find_element("xpath", '//*[@id="search-trigger-button"]/div')
    pesquisa.click()

    try:
        digite = browser.find_element("xpath", '//*[@id="search-input"]')
        digite.click()
        digite.send_keys(pais[1])
        print('Selenium: Digitou país')
    except:
        pyautogui.click(410, 137)
        pyautogui.write(pais[1])
        time.sleep(2)

    try:
        selecione = browser.find_element('xpath', '/html/body/div[6]/div[2]/div/div[2]/div/div[1]/a/div')
        selecione.click()
        selecione.click()
    except:
        pyautogui.click(377, 271)
        time.sleep(2)

    try:
        estatisticas = browser.find_element("xpath", '//*[@id="horizontally-scrollable«re8»"]/div/div/div/button[2]')
        estatisticas.click()
    except:
        pyautogui.click(860,411)

    # --- PARTE 2: CAPTURA ---
    time.sleep(1) # Espera a página carregar as estatísticas
    pyautogui.hotkey('ctrl', 'a')
    time.sleep(0.5)
    pyautogui.hotkey('ctrl', 'c')
    texto = pyperclip.paste()

    # --- PARTE 3: TRATAMENTO E SALVAMENTO (Dentro do For) ---
    try:
        # Filtra e limpa
        parte = texto.split("Geral")[1].split("Propaganda")[0]
        linhas = [l.strip() for l in parte.split('\n') if l.strip()]

        # Gera dicionário para este país específico
        stats = {}
        for i in range(0, len(linhas)-1, 2):
            chave = linhas[i]
            valor = linhas[i+1]
            stats[chave] = valor

        # Salva no banco de dados
        print(f"Salvando estatísticas de {pais[1]}...")
        salvar_estatisticas(pais[0], stats)
        print(f"Sucesso: {pais[1]} processado!")

    except IndexError:
        print(f"Erro: Não encontrou marcador 'Geral' para {pais[1]}. Verifique o idioma do site.")
    except Exception as e:
        print(f"Ocorreu um erro inesperado com {pais[1]}: {e}")

    # Pausa antes de ir para o próximo país da lista
    print("-" * 30)
    time.sleep(1)

In [ ]:
htmls = [
            browser.find_element("xpath", '/html/body/div[1]/main/div/div[1]/div[3]/div[2]/div/div[2]/div[2]').get_attribute('outerHTML'),
            # browser.find_element("xpath", '/html/body/div[1]/main/div/div[1]/div[3]/div[2]/div/div[2]/div[3]/div/div/div').get_attribute('outerHTML'),
            browser.find_element("xpath", '//*[@id="accordion-content-«r1lu»"]/div/div').get_attribute('outerHTML'),
            browser.find_element("xpath",'/html/body/div[1]/main/div/div[1]/div[3]/div[2]/div/div[2]/div[4]/div').get_attribute('outerHTML'),
            browser.find_element("xpath",'/html/body/div[1]/main/div/div[1]/div[3]/div[2]/div/div[2]/div[5]/div/div/div').get_attribute('outerHTML'),
            browser.find_element("xpath",'/html/body/div[1]/main/div/div[1]/div[3]/div[2]/div/div[2]/div[6]').get_attribute('outerHTML')
]
dados_finais = {}

for html in htmls:
    soup = BeautifulSoup(html, 'html.parser')

    spans = soup.find_all("span")
    textos = [s.get_text(strip=True) for s in spans]

    for i in range(0, len(textos), 2):
        if i + 1 < len(textos):
            chave = textos[i]
            valor = textos[i + 1]

            dados_finais[chave] = valor

print(dados_finais)